# Chapter 3 — Lexicon and Classical ML Sentiment Analysis

## Why this chapter matters

These methods turn social-media traces into careful, measurable evidence without treating posts as the whole population.

## Learning outcomes

- Apply VADER-style lexicon scoring.
- Train and evaluate TF–IDF classifiers.

## Concepts in simple language

A lexicon assigns word polarity; VADER adds rules and a compound score. Bag of words counts terms; TF–IDF downweights common terms. Unigrams are single words and bigrams are pairs. Accuracy, precision, recall and F1 reveal different errors; macro treats classes equally and weighted follows class size.

## Dataset introduction

The labelled synthetic posts support reproducible supervised learning and error analysis. All data are synthetic teaching data; they contain no real private users.

In [ ]:
from pathlib import Path
import random, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
current_path = Path.cwd()
if (current_path / 'data').exists(): repository_root = current_path
elif (current_path.parent / 'data').exists(): repository_root = current_path.parent
else: raise FileNotFoundError('Could not find the repository data directory.')
data_directory = repository_root / 'data'
print('Python:', sys.version.split()[0], '| pandas:', pd.__version__)
print('Data directory:', data_directory)


## Instructor-guided walkthrough

The next cell uses `SentimentIntensityAnalyzer`. Its output has `neg`, `neu`, `pos` proportions and a normalized `compound` score. Thresholds ±0.05 create labels.

### Function guide: `SentimentIntensityAnalyzer.polarity_scores`

**Library:** vaderSentiment  
**Purpose:** Score sentiment text.  
**Input/return:** Takes the objects shown below and returns the stated result.  
**Common mistake:** Interpreting a score as causal proof.  
**Simple example:** See the next small code cell.

| Parameter | Value used | Meaning | What happens if changed? |
|---|---:|---|---|
| text | post text | Input string | Different wording changes rule matches |
| compound threshold | ±0.05 | Maps score to label | Wider neutral band creates more neutral labels |

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer=SentimentIntensityAnalyzer(); posts=pd.read_csv(data_directory/'social_media_sample.csv').dropna(subset=['text']).drop_duplicates('post_id')
posts['vader_scores']=posts['text'].map(analyzer.polarity_scores); posts['compound']=posts['vader_scores'].map(lambda d:d['compound'])
def vader_label(score: float) -> str: return 'positive' if score>=.05 else ('negative' if score<=-.05 else 'neutral')
posts['vader_prediction']=posts['compound'].map(vader_label)
display(posts[['text','vader_scores','compound','sentiment','vader_prediction']].head())

### Function guide: `TfidfVectorizer`

**Library:** scikit-learn  
**Purpose:** Return a sparse document–term TF–IDF matrix.  
**Input/return:** Takes the objects shown below and returns the stated result.  
**Common mistake:** Interpreting a score as causal proof.  
**Simple example:** See the next small code cell.

| Parameter | Value used | Meaning | What happens if changed? |
|---|---:|---|---|
| max_features | 3000 | Vocabulary cap | Higher values use more memory |
| ngram_range | (1, 2) | Unigrams and bigrams | (1,1) removes phrases |
| min_df | 2 | Minimum document frequency | Higher values drop rare terms |
| stop_words | None | Preserve all words, including negation | 'english' may remove useful words |

### Function guide: `train_test_split`

**Library:** scikit-learn  
**Purpose:** Create reproducible training and test partitions.  
**Input/return:** Takes the objects shown below and returns the stated result.  
**Common mistake:** Interpreting a score as causal proof.  
**Simple example:** See the next small code cell.

| Parameter | Value used | Meaning | What happens if changed? |
|---|---:|---|---|
| test_size | 0.2 | 20% held out | Larger means less training data |
| random_state | 42 | Reproducible shuffle | Changing it changes the split |
| stratify | y | Preserve class proportions | None can unbalance small tests |

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score
X_train,X_test,y_train,y_test=train_test_split(posts['text'],posts['sentiment'],test_size=.2,random_state=RANDOM_SEED,stratify=posts['sentiment'])
vectorizer=TfidfVectorizer(max_features=3000,ngram_range=(1,2),min_df=2,stop_words=None)
Xtr=vectorizer.fit_transform(X_train); Xte=vectorizer.transform(X_test)
models={'Logistic regression':LogisticRegression(max_iter=500,random_state=RANDOM_SEED,class_weight='balanced'),'Multinomial NB':MultinomialNB(alpha=1.0)}
results=[]
for name,model in models.items():
    model.fit(Xtr,y_train); pred=model.predict(Xte); results.append([name,accuracy_score(y_test,pred)])
    print(name,'confusion matrix:
',confusion_matrix(y_test,pred,labels=['negative','neutral','positive']))
    print(classification_report(y_test,pred,zero_division=0))
display(pd.DataFrame(results,columns=['model','accuracy']))

`LogisticRegression` uses `max_iter=500` to allow convergence and balanced class weights. `MultinomialNB(alpha=1)` smooths unseen counts. The confusion matrix rows are true classes and columns predictions; the report summarizes precision, recall, F1 and support.

In [ ]:
best=models['Logistic regression']; pred=best.predict(Xte)
errors=pd.DataFrame({'text':X_test,'true':y_test,'predicted':pred}); display(errors[errors.true!=errors.predicted].head())

## Output interpretation

Read values in context: inspect examples, sample sizes, uncertainty, and possible bias. Association is not causation.

## Common mistakes

Common errors include ignoring missing data, leakage, class imbalance, direction of edges, and treating model confidence as research certainty.

## Challenge 1: Change VADER thresholds

**Objective:** Apply a method demonstrated above.

**Tasks:** Adapt the starter code, check the result, and write two evidence-based sentences.

**Starter code:** `# result = ...`

**Hint:** Reuse the named columns and functions from the walkthrough.

**Expected output:** A small table or metric with no missing explanation.

**Interpretation question:** What does the result support, and what can it not prove?

**Self-check:** Did you report the sample size, units, and one limitation?

## Challenge 2: Compare unigram and bigram TF–IDF models

**Objective:** Apply a method demonstrated above.

**Tasks:** Adapt the starter code, check the result, and write two evidence-based sentences.

**Starter code:** `# result = ...`

**Hint:** Reuse the named columns and functions from the walkthrough.

**Expected output:** A small table or metric with no missing explanation.

**Interpretation question:** What does the result support, and what can it not prove?

**Self-check:** Did you report the sample size, units, and one limitation?

## Challenge 3: Improve the weaker classifier

**Objective:** Apply a method demonstrated above.

**Tasks:** Adapt the starter code, check the result, and write two evidence-based sentences.

**Starter code:** `# result = ...`

**Hint:** Reuse the named columns and functions from the walkthrough.

**Expected output:** A small table or metric with no missing explanation.

**Interpretation question:** What does the result support, and what can it not prove?

**Self-check:** Did you report the sample size, units, and one limitation?

## Chapter summary

The walkthrough connected a research question to transparent data preparation, analysis, visual evidence, and cautious interpretation.

## Connection to the group project

Choose metrics before comparing models and include qualitative error analysis.